# 02 - Where the time goes

Measuring HF.Net honestly, and choosing the right path for the job.

*Dibuat oleh Gravicode Studios, dipimpin oleh Kang Fadhil.*

In [ ]:
// HF.Net is on nuget.org, so these restore straight into the notebook.
#r "nuget: Gravicode.HFNet.GraviHub, 0.3.0"
#r "nuget: Gravicode.HFNet.GraviTokenizers, 0.3.0"
#r "nuget: Gravicode.HFNet.GraviTransformers, 0.3.0"
#r "nuget: Gravicode.HFNet.GraviDatasets, 0.3.0"

using System;
using System.Linq;
#r "nuget: Gravicode.HFNet.GraviOptimum, 0.3.0"
#r "nuget: Gravicode.HFNet.GraviAccelerate, 0.3.0"

## 1. What hardware is this?

In [ ]:
using Gravicode.HFNet.GraviAccelerate;

foreach (var device in Accelerator.Devices()) Console.WriteLine(device);

The GPU is **not** the default, and that is a measured decision rather than caution.
Everything in the stack is `double`, and consumer and integrated GPUs run double-precision
arithmetic at a fraction of their single-precision rate - the foundation measured its ILGPU
path 5 to 8 times *slower* than the CPU.

## 2. Tokenizer throughput

This is where the managed implementation wins: about 1.78x faster than the Rust `tokenizers`
crate, with byte-identical output.

In [ ]:
using Gravicode.HFNet.GraviTokenizers;

var tokenizer = HfTokenizer.FromPretrained("bert-base-uncased");

string[] corpus =
[
    "Hello, world! Tokenizers are unbelievable.",
    "The capital of France is Paris, and it has been for a very long time.",
    "Quarterly revenue exceeded analyst expectations by a comfortable margin.",
    "A golden retriever played fetch in the park until the sun went down.",
];

var batch = Enumerable.Repeat(corpus, 250).SelectMany(c => c).ToList();

// Warm up past the tiered JIT recompile before timing anything.
var elapsed = Accelerator.Measure(() => tokenizer.EncodeBatch(batch),
                                  iterations: 10, warmup: 40);

Console.WriteLine($"{batch.Count:N0} documents in {elapsed.TotalMilliseconds:F1} ms");
Console.WriteLine($"{batch.Count / elapsed.TotalSeconds:N0} docs/s");

> **The warm-up count is not padding.** Tiered JIT recompiles a hot method after roughly
> thirty calls, so a measurement taken with two warm-up iterations times the interpreter's
> output. `Measure` returns the *best* run rather than the mean, because on a throttling
> laptop the mean measures the thermal state of the room.

## 3. Managed inference

The managed encoder holds its weights in float32 - exactly, since every checkpoint stores
float32 or narrower - and computes in `double`. It agrees with torch in float64 to about 1e-13,
which makes it the right tool for loading, inspecting and checking a model in pure .NET.

In [ ]:
using Gravicode.HFNet.GraviTransformers;

using var tiny = TransformerModel.Load("prajjwal1/bert-tiny");

var managed = Accelerator.Measure(() => tiny.Embed("a short sentence"),
                                  iterations: 20, warmup: 40);

Console.WriteLine($"bert-tiny, managed: {managed.TotalMilliseconds:F2} ms");

Against torch on CPU this is about 1.2x *faster* on bert-tiny and 3x slower on bert-base
(111 ms against 36 ms for one sentence). What torch still wins is the arithmetic rate of its
float32 matrix multiplies.

## 4. The production path

The same class of single-precision kernels torch uses, reached from .NET. On `bert-base-uncased`
this path takes 23.8 ms for one sentence - 1.53x faster than torch itself. The tiny model below
just keeps the download small.

In [ ]:
using Gravicode.HFNet.GraviOptimum;

const string Id = "hf-internal-testing/tiny-random-BertModel";

using var onnx = Optimum.Optimize(Id, target: "auto");
Console.WriteLine($"running on {onnx.Session.ActualTarget}");

var onnxTokenizer = HfTokenizer.FromPretrained(Id);
var feeds = Optimum.BuildEncoderInputs(onnxTokenizer, "a short sentence", onnx.Session);

Console.WriteLine(onnx.Measure(feeds, iterations: 50));

Naming a provider that is not installed **throws** rather than falling back silently - that
silent fallback is how a "CUDA" deployment runs on the CPU for months. Use `"auto"` when a
fallback is genuinely wanted.

## 5. Quantisation costs something, and it says how much

In [ ]:
// The error is measured by reading back what was written, not predicted from the format.
// bfloat16 keeps float32's exponent range; float16 has finer steps but underflows below
// about 6e-5 and loses the tail of a long-tailed weight distribution.
//
//   var report = Optimum.Quantize("model.safetensors", "model-bf16.safetensors",
//                                 QuantizationLevel.BFloat16);
//   Console.WriteLine(report);
//   // 31.9 MB -> 16.0 MB (50 %), max error 1.56E-002, mean 6.79E-005

Console.WriteLine("See docs/benchmarks.md for the full comparison against Python.");

## What to take from this

| If you are | Use |
|---|---|
| Tokenizing large volumes of text | **HF.Net** - faster than Rust, identical output |
| Loading and inspecting a checkpoint | **HF.Net** |
| Running an encoder in production | **ONNX through GraviOptimum** |
| Running an encoder to understand it | **HF.Net managed** |
| Training | **Python**, for now |

Never compare timings taken on different days - on a throttling laptop that difference
exceeds most of what is being measured.